# GPT-5 故障排除指南

GPT-5 发布以来，开发者用它构建出的各种出色成果让我们深感惊叹。我们也总结出了几类常见的故障排除模式，帮助你充分发挥该模型的能力。

## 过度思考

过度思考表现为：回答是正确的，但在处理简单请求时总响应时间不断攀升。模型不停地探索各种选项，推迟第一次工具调用，明明有简单的答案，却叙述了一段迂回曲折的过程。常见原因包括推理强度设置过高、提示词没有明确定义何时算完成，或者指导相互冲突，导致无休止的规划或引发反复慌乱的复查。 

解决这个问题的第一步是收紧 API 参数。对于常规工作，将 reasoning.effort 设置为 "minimal" 或 "low"；将更高的推理强度留给真正复杂的问题。为 assistant 提供明确的停止条件，并要求它在回复前进行一次快速的自检。可以考虑使用 gpt-5-mini 或 nano 对用户请求进行分类，并以合适的推理强度设置进行路由。如果收集上下文是任务的一部分，请告诉模型收集回答所需数据的最佳实践。

```
<efficient_context_understanding_spec>
Goal: Get enough context fast and stop as soon as you can act.

Method:
- Start broad, then fan out to focused subqueries.
- In parallel, launch 4–8 varied queries; read top 3–5 hits per query. Deduplicate paths and cache; don't repeat queries.

Early stop (act if any):
- You can name exact files/symbols to change.
- You can repro a failing test/lint or have a high-confidence bug locus.
</efficient_context_understanding_spec>
```

下面的示例与此类似，不同之处在于它指示模型对于无需调查或工具调用的问题立即作答，而不是过度思考。

```
# Fast-path for trivial Q&A (latency optimization)
Use this section ONLY when the user's question:
- Is general knowledge or a simple usage query
- Requires no commands, browsing, or tool calls
- Especially if the user is asking an informational question or how to perform a task, rather than asking you to run that task, provide concise instructions about how the user can do it.

Exceptions:
- If the question references files/paths/functions, requests execution/verifications, or needs more context, use the normal flow
- If unsure whether fast-path applies, ask one brief clarifying question; otherwise proceed with normal flow

Behavior:
- Answer immediately and concisely
- No status updates, no todos, no summaries, no tool calls
- Ignore the rest of the instructions following this section and simply respond right away.
```

## 偷懒/思考不足

在使用 gpt-5 时，你可能遇到过模型在给出答案前没有花足够时间推理而导致的失败。 

按照我们的最佳实践，有 2 种缓解方法：

1. 使用更高的 `reasoning_effort`：`reasoning_effort` 参数控制模型思考的程度以及调用工具的积极性。如果你之前使用的是 `minimal`，可以尝试 `low`；如果之前是 `low`，可以尝试 `medium`，依此类推。  
2. 通过提示词鼓励模型自我反思并为自己的回答打分。例如，让模型构建一套内部评分标准，并在回复前将其应用到解决方案上，这在编码任务中效果出奇地好。你也可以提供自己的评分标准，并指示模型反思自己的工作，在回复前如发现问题就进行迭代。

```
<self_reflection>
- Internally score the draft against a 5–7 item rubric you devise (clarity, correctness, edge cases, completeness, latency).
- If any category falls short, iterate once before replying.
</self_reflection>
```

## 过于顺从

GPT-5 有时会过于顺从。尤其是在智能体场景中，我们往往希望模型能自主行动、“直接把事情做完”。在系统提示词中提供坚持执行的指令，可以有效缓解这种行为。使用更高的 `reasoning_effort`（`low` 及以上）时，这种引导会更容易生效。

```
<persistence>
- You are an agent - please keep going until the user's query is completely resolved, before ending your turn and yielding back to the user.
- Only terminate your turn when you are sure that the problem is solved.
- Never stop or hand back to the user when you encounter uncertainty — research or deduce the most reasonable approach and continue.
- Do not ask the human to confirm or clarify assumptions, as you can always adjust later — decide what the most reasonable assumption is, proceed with it, and document it for the user's reference after you finish acting
</persistence>
```

## 过于冗长

GPT-5 在给用户的最终消息中，有时会生成比你期望更多的 token。

有两种简单的方法可以解决这个问题。第一种是在 API 中调低 `verbosity` 参数。如果未指定，冗长度默认为 `medium`，因此如果你想要更简短的输出，可以尝试显式地将其设为 `low`。第二种是在系统提示词中进行设置，这在编码场景中尤其有效

```
Write code for clarity first. Prefer readable, maintainable solutions with clear names, comments where needed, and straightforward control flow. Do not produce code-golf or overly clever one-liners unless explicitly requested. Use high verbosity for writing code and code tools.
```

## 延迟

延迟由几个不同的因素构成，因此在调优之前一定要先进行测量。跟踪 P50/P95 下的 TTFT（首 token 时间）、首次行动时间和总响应时间，并将模型时间与工具和网络时间区分开。跟踪这些指标可以帮助你优化真正慢的那个环节。

要缩短模型响应时间，请合理控制模型的思考量：常规工作使用 reasoning.effort "minimal" 或 "low"，并添加明确的停止条件和一次性自检（参见“过度思考”）。更高的推理强度也可能导致更多的工具调用。

尽可能合并工具调用。你需要告诉模型何时并行调用工具，默认情况下它并不总会这样做。 

```
<parallelization_spec>
Definition: Run independent or read-only tool actions in parallel (same turn/batch) to reduce latency.
When to parallelize:
 - Reading multiple files/configs/logs that don’t affect each other.
 - Static analysis, searches, or metadata queries with no side effects.
 - Separate edits to unrelated files/features that won’t conflict.
</parallelization_spec>
```

为了让用户能够[在模型推理时查看进度](https://platform.openai.com/docs/guides/latency-optimization#make-your-users-wait-less)，可以向用户展示推理摘要和工具调用前言消息。在很多情况下，当模型思考时如果不向用户展示推理摘要，感知延迟反而会降低。你还可以指示模型在调用工具前提供前言消息或状态更新，让用户在模型调用工具时了解它在做什么。

```
<status_update_spec>
Definition: A brief progress note: what just happened, what’s next, any real blockers, written in a continuous conversational style, narrating the story of your progress as you go.
Always start with a brief acknowledgement of the task before getting started. (No need to prefix with "Status Update:")
</status_update_spec>
```

通过缓存不变的内容来降低 TTFT：合理组织发往 API 的请求，确保有效利用提示词、推理和工具调用结果的缓存。当某条路径确实对延迟敏感时，可以为该调用启用优先处理 `service_tier = “priority”` 以获得更快的响应（注意，优先处理服务的 token 按 token 计费，价格高于标准处理费率）。如果在提示词很短且没有工具的情况下 TTFT 仍然很高，请保存 `request_id` 并上报至 [support@openai.com](mailto:support@openai.com)，以获得更有针对性的帮助。

## 调用过多工具

当模型频繁调用工具却没有推进答案时，常见原因是路由模糊：工具定义相互重叠、提示词更看重全面而非果断，或者推理强度设置过高。另一个常见原因是没有将先前的推理带入后续调用；使用 Responses API 可以确保意图和推理摘要在多轮之间得以保留，而不会忘记当初为什么选择某个工具。

在提示词指令中，将基于上下文直接作答设为默认行为。让每个工具只承担单一职责，具备清晰的输入/输出，并明确注明“不要用于……”。为常见场景提供简短的操作手册，让路径一目了然（例如：如果用户提到了一份你上下文中没有的文档，先运行语义搜索找到它，然后获取相关章节再作答）。 

```
<tool_use_policy>
Select one tool or none; prefer answering from context when possible.
Cap tool calls at 2 per user request unless new information makes more strictly necessary.
</tool_use_policy>
```

关注 tool\_calls\_per\_turn、几秒内对同一工具的重复调用，以及无需工具即完成的回答所占比例；这些指标出现激增，就明确表明路由或提示词需要收紧。

## 格式错误的工具调用

在极少数情况下，gpt-5 可能出现模式崩溃（mode collapse），即模型调用工具时输出一长串重复的无意义内容。 

在这些情况下，我们发现问题总是源于提示词不同部分之间的矛盾。作为最佳实践，我们建议利用 gpt-5 的元提示（meta prompting）能力来定位并修复该问题

```
Please analyze why the <tool_name> tool call is malformed.
1. Review the provided sample issue to understand the failure mode.
2. Examine the <System Prompt> and <Tool Config> carefully. Identify any ambiguities, inconsistencies, or phrasing that could mislead GPT-5 into generating an incorrect tool call.
3. For each potential cause, explain clearly how it could result in the observed failure.
4. Provide actionable recommendations to improve the <System Prompt> or <Tool Config> so GPT-5 produces valid tool calls consistently.


<System Prompt>

<Tool Config>
```

## 通用故障排除 

上面许多提示词补充内容都是通过元提示生成的。在某一轮表现未达预期时，可以在该轮结束后询问 GPT-5 如何改进它自己的指令。下面的提示词曾用于生成上文中针对过度思考问题的部分解决方案，你可以根据自己的具体需求进行修改。

```
That was a high quality response, thanks! It seemed like it took you a while to finish responding though. Is there a way to clarify your instructions so you can get to a  response as good as this faster next time? It's extremely important to be efficient when providing these responses or users won't get the most out of them in time. Let's see if we can improve!
1) think through the response you gave above
2) read through your instructions starting from "<insert the first line of the system prompt here>" and look for anything that might have made you take longer to formulate a high quality response than you needed
3) write out targeted (but generalized) additions/changes/deletions to your instructions to make a request like this one faster next time with the same level of quality
```

在特定上下文中进行元提示时，如有可能，最好多生成几次回答，并留意这些回答中的共同元素。模型提出的一些改进或修改可能过于针对当时的特定情况，但你通常可以将其简化，得出通用的改进方案。我们建议你创建一个 eval，来衡量某项提示词修改对你的特定用例是更好还是更差。